# Interne en externe links
Dit notebook ondersteunt het verkennen, analyseren van klik functionaliteit binnen AskDelphi.

### Connectie met AskDelphi opzetten

In [ ]:
import pprint
from ask_delphi_api.topictools import TopicTools
from ask_delphi_api.import_digicoach import Import

import_service = Import()
topic_tools = TopicTools(import_service.client, import_service.project)

### Extractie proces, taken, stappen uit het taaksjabloon

In [ ]:
from ask_delphi_api.convert_taaksjabloon import read_dir, convert_doc_to_json
import pprint
import json
from ask_delphi_api.import_digicoach import Import

paths = read_dir('/Users/baasa03/Digicoach/taaksjabloon_demo')
json_coaches = []

for path in paths:
    try:
        print("\n\n")
        json_coach = convert_doc_to_json(path)
        json_coaches.append(json_coach)
    except ValueError as e:
        print(e)

json_digicoach = json.loads(json_coaches[0])

### Registratie bronnen uit taaksjabloon

In [ ]:
import_service = Import()

# Ophalen bronnen lijst taaksjabloon
sources = json_digicoach["sources"]

# Aanmaken sources
import_service.create_source_topics(sources)

# Ophalen bijgewerkte topic lijst
topics = topic_tools.fetch_topiclist()

# ['Questionnaire', 'Image map', 'Profile details portlet', 'Expandable items', 'My profile', 
#  'Tabbed portlet', 'Skill area', 'ConnectPeople', 'SharePoint', 'Gallery', 'Micropolling definition', 
#  'Blob', 'Publication Reference', 'Pre-defined search', 'Uitgeverij extern', 'Homepage', 'Cornerstone', 
#  'Flipcard Portlet', 'Concept', 'Btube Link', 'Task', 'Digitale Coach Procespagina', 'External URL', 
#  'Intranet', 'Hierarchy', 'Rakoo', 'Moodle', 'Person', 'Action', 'Pagina Structuur Voorgedefinieerde Zoekopdracht', 
#  'Collection', 'Video', 'Quick question', 'Menu', 'Image', 'Glossary item', 'Digitale Coach Homepagina', 'Simple topic', 'Sketchbook']"

### Topic IDs interne en externe links

In [ ]:
import_service.create_link_list(json_digicoach, topics)
pprint.pprint(import_service.link_list)

### Test creatie Externe link (Intranet)

In [ ]:
link = import_service.create_link("KRB Klantbeeld", "2e04df19-54fa-4341-be5f-0d9c5d1d1635")
pprint.pprint(link)

### Test zoek bronverwijzingen en vervang deze door een hyperlink

In [ ]:
# text = "Deze tekst gaat over Kwijtscheldingsformulier ondernemers in 2026. Ga naar Verzoek beoordelen"

text = """Je gaat de vermogensbestanddelen beoordelen in relatie met de openstaande schuld waar sanering voor wordt gevraagd. 
Let op: Het moet de formele belastingschuld betreffen. Zie artikel 26.3.4 Leidraad Invordering 2008 en paragraaf 43.4.1 IIB. 
Vermogensbestanddelen Log in Inzicht en kies voor de knop “invordering”. Vul daar het BSN/RSIN in. Je komt dan in het klantprofiel. 
Linksboven zie je drie streepjes waar je op klikt nu kun je verschillende optie aanklikken. 
Voor de invordering is vooral van belang de knop “invordering” hier zie je alle vermogensbestanddelen die bij de Belastingdienst bekend zijn LET OP: 
De gegevens met betrekking tot panden is niet volledig. Hiervoor moet je altijd het Kadaster (laten) raadplegen. 
Je gaat vervolgens RBG raadplegen. Daar kun je zien wat voor banksaldo’s er waren in voorgaande jaren. 
In RBG, kies “zoek rekeninghouders”, Toets het RSIN/BSN in en kies een jaartal. 
Per jaartal moet je steeds opnieuw het RSIN/BSN intoetsen. De saldo’s in RBG zijn de saldo’s aan het eind van een jaar. 
Vermogen in het verleden is geen afwijsgrond onder tijdelijke instructie saneringen. 
Hou hier rekening mee bij het opvragen van aanvullende stukken en neem altijd ook contact op met de ondernemer. 
Als je tot de conclusie komt dat er voldoende vermogen is en/of was tot aan het verzoek, 
wijs het verzoek dan af met as reden dat er vanaf de bekendmaking van de belastingaanslag tot aan het indiening van het verzoek 
om sanering op enig moment voldoende middelen aanwezig waren om de aanslag(en) te kunnen voldoen. 
Zie ook de werkinstructies Beoordelen Vermogensbestanddelen"""

content = import_service.hyperlink_html(text)
pprint.pprint(content)

### Update content "Open postbak in WAB"

In [ ]:
text = "Deze tekst gaat over Kwijtscheldingsformulier ondernemers in 2026. Ga naar Verzoek beoordelen"
topic_id = "04c5a67f-7b79-4df3-a3e5-ad734c8b18ce"
topic_version_id = topic_tools.get_topicVersionId(topic_id)
import_service.add_content_to_topic(topic_id, topic_version_id, text)
topic_tools.checkin(topic_id)
import_service.publiceer(topic_id)

### Code om structuur uit te vragen

In [ ]:
# import pprint

# # Haal topic-parts op.
# # topicId = "1ef44c28-9974-4899-b29a-fe352c9128d6"
# topicId = "b78eb9d8-1a27-4bbb-a542-3102e39deb86"
# content = topic_tools.get_topic_parts(topicId=topicId)
# body_part = ""

# # Selecteer part uit topic met daarin de content.
# body_part = None
# groups = content['topicEditorData']['groups']
# for group in groups:
#     for part in group['parts']:
#         if part["partId"] == "body":
#             body_part = part['editors'][0]['value']['richTextEditor']['value']

# pprint.pp(body_part)

### Toevoegen link aan pyramide

In [ ]:
topic_taak = import_service.topic.get_topic_by_title("Selecteer Zaak",topics)
topic_id_taak = topic_taak["topicGuid"]
print(f"topic_taak : {topic_taak}")

topic_link = import_service.topic.get_topic_by_title("Kwijtscheldingsformulier Ondernemers",topics)
topic_id_link = topic_link["topicGuid"]
print(f"topic_link : {topic_link}")

import_service.topic.checkout(topic_id_taak)

topic_version_id_taak = topic_tools.get_topicVersionId(topic_id_taak)

def keys_by_value(value):    
    return [k for k, v in import_service.link_list.items() if v == value]

# Pyramide bijwerken
def add_sources(topic_id: str, topic_version_id: str, text: str, sources: list[dict]):

    topic_id_links = []

    # Externe linkjes waarvan de titel voorkomt in de text detecteren
    for source in sources:
        # print(f"{source["titel"]}, {source["type"]}, {source["link"]}")
        if source["titel"] in text:
            topic_id_link = import_service.link_list[source["titel"]]
            topic_id_links.append(topic_id_link)

    # RelationTypeId "Handleidingen en instructies" uitvragen
    # Todo : In eerste instantie de links onder Handleidingen en instructies geplaatst, navraag hoe dit te verbeteren
    relationTypeId = import_service.relation.get_relationTypeId_by_relationTypeName(topic_id, topic_version_id, "Handleidingen en instructies")

    # Externe links als relatie in de pyramide toevoegen
    for topic_id_link in topic_id_links:
        # import_service.relation.add_relation(topic_id, topic_version_id, relationTypeId, topic_id_link)
        link_title = keys_by_value(topic_id_link)
        print(f"Externe link : {link_title} toegevoegd onder Handleidingen en instructies")

text = "Deze tekst gaat over Kwijtscheldingsformulier Ondernemers in 2026. Ga naar Verzoek beoordelen. Beoordelen Vermogensbestanddelen"
add_sources(topic_id_taak, topic_version_id_taak, text, sources)

import_service.topic.checkin(topic_id_taak)
import_service.publiceer(topic_id_taak)
